# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

In the comment at the start of certain blocks, I note some specific settings to consider modifying.

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.

In [4]:
# Installs microWakeWord. Be sure to restart the session after this is finished.
import platform

if platform.system() == "Darwin":
    # `pymicro-features` is installed from a fork to support building on macOS
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# `audio-metadata` is installed from a fork to unpin `attrs` from a version that breaks Jupyter
!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

!git clone https://github.com/kahrendt/microWakeWord
!pip install -e ./microWakeWord

  Cloning https://github.com/whatsnowplaying/audio-metadata (to revision d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f) to /tmp/pip-req-build-wdimg90w
  Running command git clone --filter=blob:none --quiet https://github.com/whatsnowplaying/audio-metadata /tmp/pip-req-build-wdimg90w
  Running command git rev-parse -q --verify 'sha^d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
  Running command git fetch -q https://github.com/whatsnowplaying/audio-metadata d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Running command git checkout -q d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Resolved https://github.com/whatsnowplaying/audio-metadata to commit d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cloning into 'microWakeWord'...
fatal: could not read Username for 'https://github.com': No such device or address
ERROR: ./microWakeWord is not a valid editable requirement. It should e

In [5]:
import os
import sys
import glob
import shutil

print("Python:", sys.executable)

# Cài package cần thiết
!{sys.executable} -m pip install -q piper-sample-generator piper-phonemize-cross==1.2.1 onnxruntime cython numpy

# Lấy source piper để có piper_train
%cd /content

if not os.path.exists("/content/piper"):
    !git clone --depth 1 https://github.com/rhasspy/piper.git /content/piper

# Cài piper_train từ source, không kéo dependency cũ
!{sys.executable} -m pip install -q --no-deps -e /content/piper/src/python

# Build monotonic_align
!cd /content/piper/src/python/piper_train/vits/monotonic_align && {sys.executable} setup.py build_ext --inplace

# Copy file .so vào đúng thư mục import
base = "/content/piper/src/python/piper_train/vits/monotonic_align"
dst = os.path.join(base, "monotonic_align")
os.makedirs(dst, exist_ok=True)

so_files = glob.glob(base + "/build/lib*/piper_train/vits/monotonic_align/core*.so")
print("Found .so files:", so_files)

if so_files:
    src = so_files[0]
    dst_file = os.path.join(dst, os.path.basename(src))
    shutil.copy2(src, dst_file)
    open(os.path.join(dst, "__init__.py"), "a").close()
    print("Copied:", dst_file)

# Test import
sys.path.insert(0, "/content/piper/src/python")

from piper_train.vits import commons
from piper_train.vits.monotonic_align import maximum_path_c

print("piper_train + monotonic_align OK")

Python: /usr/bin/python3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 25.8 MB/s eta 0:00:00
/content
Cloning into '/content/piper'...
remote: Enumerating objects: 254, done.
remote: Counting objects: 100% (254/254), done.
remote: Compressing objects: 100% (233/233), done.
remote: Total 254 (delta 10), reused 184 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (254/254), 24.45 MiB | 31.17 MiB/s, done.
Resolving deltas: 100% (10/10), done.
  Preparing metadat

In [6]:
!mkdir -p voices

!curl -fL --retry 3 -o voices/en_US-lessac-medium.onnx \
  "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx?download=true"

!curl -fL --retry 3 -o voices/en_US-lessac-medium.onnx.json \
  "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx.json?download=true"

!ls -lh voices

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1024  100  1024    0     0   6543      0 --:--:-- --:--:-- --:--:--  6564
100 60.2M  100 60.2M    0     0  54.3M      0  0:00:01  0:00:01 --:--:--  145M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   249  100   249    0     0   1191      0 --:--:-- --:--:-- --:--:--  1197
100  4885  100  4885    0     0  22362      0 --:--:-- --:--:-- --:--:-- 22362
total 61M
-rw-r--r-- 1 root root  61M Jul  8 00:35 en_US-lessac-medium.onnx
-rw-r--r-- 1 root root 4.8K Jul  8 00:35 en_US-lessac-medium.onnx.json


In [7]:
import os
import sys
import subprocess
from IPython.display import Audio

target_word = "hi luminova"
print("Target word:", target_word)

os.system("rm -rf generated_samples")
os.makedirs("generated_samples", exist_ok=True)

env = os.environ.copy()
env["PYTHONPATH"] = "/content/piper/src/python:" + env.get("PYTHONPATH", "")

cmd = [
    sys.executable, "-m", "piper_sample_generator",
    target_word,
    "--model", "voices/en_US-lessac-medium.onnx",
    "--max-samples", "1",
    "--batch-size", "1",
    "--output-dir", "generated_samples",
    "--length-scales", "1.0",
    "--noise-scales", "0.667",
    "--noise-scale-ws", "0.8",
]

print("CMD:", " ".join(cmd))

result = subprocess.run(cmd, env=env, capture_output=True, text=True)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

!ls -lh generated_samples

Audio(filename="generated_samples/0.wav", autoplay=True)

Target word: hi luminova
CMD: /usr/bin/python3 -m piper_sample_generator hi luminova --model voices/en_US-lessac-medium.onnx --max-samples 1 --batch-size 1 --output-dir generated_samples --length-scales 1.0 --noise-scales 0.667 --noise-scale-ws 0.8
Return code: 0

DEBUG:__main__:Loading ['voices/en_US-lessac-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: voices/en_US-lessac-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)

total 48K
-rw-r--r-- 1 root root 45K Jul  8 00:35 0.wav


In [8]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

MODEL_PATH = "voices/en_US-lessac-medium.onnx"

variants = [
    ("hey lu mi no va", 500),
    ("hey loo mee no va", 250),
    ("hey lu mi noh va", 150),
    ("hey luminova", 100),
]

os.system("rm -rf generated_samples generated_variant_tmp")
os.makedirs("generated_samples", exist_ok=True)
os.makedirs("generated_variant_tmp", exist_ok=True)

env = os.environ.copy()
env["PYTHONPATH"] = "/content/piper/src/python:" + env.get("PYTHONPATH", "")

global_index = 0

for text, count in variants:
    tmp_dir = f"generated_variant_tmp/{text.replace(' ', '_')}"
    os.makedirs(tmp_dir, exist_ok=True)

    print("Generating:", text, "count:", count)

    cmd = [
        sys.executable, "-m", "piper_sample_generator",
        text,
        "--model", MODEL_PATH,
        "--max-samples", str(count),
        "--batch-size", "100",
        "--output-dir", tmp_dir,
        "--length-scales", "0.9", "1.0", "1.1",
        "--noise-scales", "0.667",
        "--noise-scale-ws", "0.8",
    ]

    result = subprocess.run(cmd, env=env, capture_output=True, text=True)

    print("Return code:", result.returncode)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)

    wav_files = sorted(Path(tmp_dir).glob("*.wav"))

    for wav_file in wav_files:
        dst = f"generated_samples/{global_index}.wav"
        shutil.copy2(wav_file, dst)
        global_index += 1

print("Total generated samples:")
!find generated_samples -type f | wc -l

Generate wake word: hi luminova
CMD: /usr/bin/python3 -m piper_sample_generator hi luminova --model voices/en_US-lessac-medium.onnx --max-samples 1000 --batch-size 100 --output-dir generated_samples --length-scales 1.0 --noise-scales 0.667 --noise-scale-ws 0.8
Return code: 0

DEBUG:__main__:Loading ['voices/en_US-lessac-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: voices/en_US-lessac-medium.onnx.json
DEBUG:piper.voice:Using CUDA
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
INFO:__main__:Successfully loaded model(s)

Total samples:
1000


In [9]:
from IPython.display import Audio, display
import random
import os

for i in random.sample(range(1000), 5):
    path = f"generated_samples/{i}.wav"
    print(path, os.path.exists(path))
    display(Audio(filename=path, autoplay=False))

generated_samples/946.wav True


generated_samples/715.wav True


generated_samples/629.wav True


generated_samples/758.wav True


generated_samples/460.wav True


In [11]:
!zip -q -r hi_luminova_generated_samples.zip generated_samples

from google.colab import files
files.download("hi_luminova_generated_samples.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import os
from pathlib import Path
from tqdm import tqdm
import numpy as np
import scipy.io.wavfile
import librosa

# Dọn dữ liệu cũ
!rm -rf fma fma_16k mit_rirs audioset audioset_16k

# Tạo thư mục rỗng
!mkdir -p fma fma_16k mit_rirs audioset_16k

# Tải FMA noise/background
!wget -O fma/fma_xs.zip "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip"

# Giải nén
!cd fma && unzip -q fma_xs.zip

# Convert mp3 -> wav 16kHz mono
mp3_files = list(Path("fma/fma_small").glob("**/*.mp3"))
print("MP3 files:", len(mp3_files))

for mp3_path in tqdm(mp3_files):
    y, sr = librosa.load(str(mp3_path), sr=16000, mono=True)
    y = np.clip(y, -1.0, 1.0)
    wav = (y * 32767).astype(np.int16)

    out_name = mp3_path.stem + ".wav"
    scipy.io.wavfile.write(os.path.join("fma_16k", out_name), 16000, wav)

print("fma_16k files:")
!find fma_16k -type f | wc -l

print("generated_samples:")
!find generated_samples -type f | wc -l

--2026-07-08 00:45:10--  https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip
Resolving huggingface.co (huggingface.co)... 3.163.189.90, 3.163.189.114, 3.163.189.37, ...
Connecting to huggingface.co (huggingface.co)|3.163.189.90|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/66ca2060627175be3322dcd0/b15b33649259980016c036a0b2a0fee516e0a9acf3f1adcc561e0abd5efde02b?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27fma_xs.zip%3B+filename%3D%22fma_xs.zip%22%3B&response-content-type=application%2Fzip&user_id=public&X-Xet-Cas-Uid=public&Expires=1783475110&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjZjYTIwNjA2MjcxNzViZTMzMjJkY2QwL2IxNWIzMzY0OTI1OTk4MDAxNmMwMzZhMGIyYTBmZWU1MTZlMGE5YWNmM2YxYWRjYzU2MWUwYWJkNWVmZGUwMmJcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlubGluZSUzQitmaWxlbmFtZSUyQSUzRFVURi04JTI3JTI3Zm1hX3hzLnppcCUzQitmaWxlbmFtZSUzRCUy

100%|██████████| 210/210 [00:33<00:00,  6.20it/s]

fma_16k files:
210


generated_samples:
1000


In [13]:
import os
import shutil

output_dir = "./negative_datasets"

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.mkdir(output_dir)

link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
filenames = [
    "dinner_party.zip",
    "dinner_party_eval.zip",
    "no_speech.zip",
    "speech.zip",
]

for fname in filenames:
    link = link_root + fname
    zip_path = f"negative_datasets/{fname}"

    print("Downloading:", fname)
    !wget -O {zip_path} {link}

    print("Unzipping:", fname)
    !unzip -q {zip_path} -d {output_dir}

print("negative_datasets files:")
!find negative_datasets -type f | wc -l

Downloading: dinner_party.zip
--2026-07-08 00:46:07--  https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/dinner_party.zip
Resolving huggingface.co (huggingface.co)... 3.163.189.37, 3.163.189.114, 3.163.189.90, ...
Connecting to huggingface.co (huggingface.co)|3.163.189.37|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/65e327bc1445a768ed343b8c/228d7e72cd5fdc4e6e57da36b88a4c227d34cb8dc44041078b4c4b65dc75848d?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27dinner_party.zip%3B+filename%3D%22dinner_party.zip%22%3B&response-content-type=application%2Fzip&user_id=public&Expires=1783475167&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjVlMzI3YmMxNDQ1YTc2OGVkMzQzYjhjLzIyOGQ3ZTcyY2Q1ZmRjNGU2ZTU3ZGEzNmI4OGE0YzIyN2QzNGNiOGRjNDQwNDEwNzhiNGM0YjY1ZGM3NTg0OGRcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPWlu

In [14]:
import audiomentations

class DummyAddColorNoise:
    def __init__(self, *args, **kwargs):
        pass

    def __call__(self, samples, sample_rate=None, **kwargs):
        return samples

audiomentations.AddColorNoise = DummyAddColorNoise

print("Patch OK:", hasattr(audiomentations, "AddColorNoise"))

Patch OK: True


In [16]:
import os
import sys
import glob
import subprocess

os.chdir("/content")

repo = "/content/microWakeWord"

# Clone lại repo nếu chưa có
if not os.path.exists(repo):
    subprocess.check_call([
        "git", "clone",
        "https://github.com/OHF-Voice/micro-wake-word",
        repo
    ])

# Cài lại đúng vào Python notebook hiện tại
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-e", repo
])

# Cài các dependency cần cho các cell sau
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "mmap-ninja",
    "pyyaml",
    "audiomentations",
    "librosa",
    "soundfile",
    "scipy",
    "tqdm",
    "numpy"
])

# Ép Python nhìn thấy repo
if repo not in sys.path:
    sys.path.insert(0, repo)

# Tìm thư mục module thật
candidates = glob.glob("/content/**/microwakeword", recursive=True)
print("Found microwakeword dirs:")
for c in candidates:
    print(c)
    parent = os.path.dirname(c)
    if parent not in sys.path:
        sys.path.insert(0, parent)

import microwakeword
print("microWakeWord import OK:", microwakeword.__file__)

Found microwakeword dirs:
/content/microWakeWord/microwakeword
microWakeWord import OK: /content/microWakeWord/microwakeword/__init__.py


In [17]:
!echo "generated_samples:"
!find generated_samples -type f 2>/dev/null | wc -l

!echo "fma_16k:"
!find fma_16k -type f 2>/dev/null | wc -l

!echo "negative_datasets:"
!find negative_datasets -type f 2>/dev/null | wc -l

generated_samples:
1000
fma_16k:
210
negative_datasets:
521


In [18]:
import audiomentations

class DummyAddColorNoise:
    def __init__(self, *args, **kwargs):
        pass

    def __call__(self, samples, sample_rate=None, **kwargs):
        return samples

audiomentations.AddColorNoise = DummyAddColorNoise

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(
    input_directory="generated_samples",
    file_pattern="*.wav",
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        "SevenBandParametricEQ": 0.1,
        "TanhDistortion": 0.1,
        "PitchShift": 0.1,
        "BandStopFilter": 0.1,
        "AddColorNoise": 0.0,
        "AddBackgroundNoise": 0.75,
        "Gain": 1.0,
    },
    impulse_paths=[],
    background_paths=["fma_16k"],
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

print("Augmentation setup OK")

Augmentation setup OK


In [19]:
from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, "augmented_clip.wav")

Audio(filename="augmented_clip.wav", autoplay=True)

In [20]:
import os
import shutil
from mmap_ninja.ragged import RaggedMmap

output_dir = "generated_augmented_features"

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

os.mkdir(output_dir)

splits = ["training", "validation", "testing"]

for split in splits:
    out_dir = os.path.join(output_dir, split)
    os.mkdir(out_dir)

    split_name = "train"
    repetition = 2

    spectrograms = SpectrogramGeneration(
        clips=clips,
        augmenter=augmenter,
        slide_frames=10,
        step_ms=10,
    )

    if split == "validation":
        split_name = "validation"
        repetition = 1

    elif split == "testing":
        split_name = "test"
        repetition = 1
        spectrograms = SpectrogramGeneration(
            clips=clips,
            augmenter=augmenter,
            slide_frames=1,
            step_ms=10,
        )

    print("Generating:", split)

    RaggedMmap.from_generator(
        out_dir=os.path.join(out_dir, "wakeword_mmap"),
        sample_generator=spectrograms.spectrogram_generator(
            split=split_name,
            repeat=repetition,
        ),
        batch_size=100,
        verbose=True,
    )

print("Generated augmented features OK")

Generating: training


0it [00:00, ?it/s]

Generating: validation


0it [00:00, ?it/s]

Generating: testing


0it [00:00, ?it/s]

Generated augmented features OK


In [21]:
import yaml
import os
import shutil

config = {}

config["window_step_ms"] = 10
config["train_dir"] = "trained_models/hi_luminova_light"

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 6.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 4.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 3.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
]

config["training_steps"] = [1000]
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]
config["learning_rates"] = [0.001]
config["batch_size"] = 32

config["time_mask_max_size"] = [0]
config["time_mask_count"] = [0]
config["freq_mask_max_size"] = [0]
config["freq_mask_count"] = [0]

# Quan trọng: lớn hơn training_steps để tránh eval giữa chừng gây spike RAM
config["eval_step_interval"] = 2000

config["clip_duration_ms"] = 1500
config["target_minimization"] = 0.9
config["minimization_metric"] = None
config["maximization_metric"] = "average_viable_recall"

with open("training_parameters_hi_luminova_light.yaml", "w") as f:
    yaml.dump(config, f)

if os.path.exists("trained_models/hi_luminova_light"):
    shutil.rmtree("trained_models/hi_luminova_light")

print("Saved training_parameters_hi_luminova_light.yaml")

!cat training_parameters_hi_luminova_light.yaml | grep -E "train_dir|training_steps|batch_size|eval_step_interval" -A 2

Saved training_parameters_hi_luminova_light.yaml
batch_size: 32
clip_duration_ms: 1500
eval_step_interval: 2000
features:
- features_dir: generated_augmented_features
--
train_dir: trained_models/hi_luminova_light
training_steps:
- 1000
window_step_ms: 10


In [22]:
import sys
import subprocess
import os

env = os.environ.copy()
env["PYTHONPATH"] = "/content/microWakeWord:" + env.get("PYTHONPATH", "")
env["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

cmd = [
    sys.executable, "-m", "microwakeword.model_train_eval",
    "--training_config=training_parameters_hi_luminova_light.yaml",
    "--train", "1",
    "--restore_checkpoint", "0",
    "--test_tf_nonstreaming", "0",
    "--test_tflite_nonstreaming", "0",
    "--test_tflite_nonstreaming_quantized", "0",
    "--test_tflite_streaming", "0",
    "--test_tflite_streaming_quantized", "1",
    "--use_weights", "best_weights",
    "mixednet",
    "--pointwise_filters", "64,64,64,64",
    "--repeat_in_block", "1, 1, 1, 1",
    "--mixconv_kernel_sizes", "[5], [7,11], [9,15], [23]",
    "--residual_connection", "0,0,0,0",
    "--first_conv_filters", "32",
    "--first_conv_kernel_size", "5",
    "--stride", "3",
]

print("Running:")
print(" ".join(cmd))

process = subprocess.run(cmd, env=env, text=True)

print("Return code:", process.returncode)

Running:
/usr/bin/python3 -m microwakeword.model_train_eval --training_config=training_parameters_hi_luminova_light.yaml --train 1 --restore_checkpoint 0 --test_tf_nonstreaming 0 --test_tflite_nonstreaming 0 --test_tflite_nonstreaming_quantized 0 --test_tflite_streaming 0 --test_tflite_streaming_quantized 1 --use_weights best_weights mixednet --pointwise_filters 64,64,64,64 --repeat_in_block 1, 1, 1, 1 --mixconv_kernel_sizes [5], [7,11], [9,15], [23] --residual_connection 0,0,0,0 --first_conv_filters 32 --first_conv_kernel_size 5 --stride 3
Return code: 0


In [23]:
!find trained_models -name "*.tflite" -print

trained_models/hi_luminova_light/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite


In [24]:
import os
import json
import shutil

model_src = "trained_models/hi_luminova_light/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
model_dst = "hi_luminova.tflite"

if not os.path.exists(model_src):
    raise FileNotFoundError(model_src)

shutil.copy2(model_src, model_dst)

manifest = {
    "type": "micro",
    "wake_word": "Hi Luminova",
    "author": "Luminova",
    "website": "",
    "model": "hi_luminova.tflite",
    "trained_languages": ["en"],
    "version": 2,
    "micro": {
        "probability_cutoff": 0.95,
        "sliding_window_size": 5,
        "feature_step_size": 10,
        "tensor_arena_size": 50000
    }
}

with open("hi_luminova.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

!ls -lh hi_luminova.tflite hi_luminova.json

-rw-r--r-- 1 root root 320 Jul  8 01:03 hi_luminova.json
-rw-r--r-- 1 root root 60K Jul  8 01:02 hi_luminova.tflite


In [25]:
from google.colab import files

files.download("hi_luminova.tflite")
files.download("hi_luminova.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>